# RAIL Score SDK — Compliance Checks

Evaluate AI-generated content against regulatory frameworks:
- **GDPR** (EU data protection)
- **CCPA** (California privacy)
- **HIPAA** (US healthcare)
- **EU AI Act** (risk-based AI regulation)
- **India DPDP** (digital personal data protection)
- **India AI Gov** (AI governance)

Supports single-framework, multi-framework, and strict mode.

**Prerequisites:** `pip install rail-score-sdk`

```bash
export RAIL_API_KEY="rail_..."
```

In [ ]:
import os

RAIL_API_KEY = os.environ.get("RAIL_API_KEY")
assert RAIL_API_KEY, "Set RAIL_API_KEY environment variable before running this notebook"

In [ ]:
from rail_score_sdk import RailScoreClient

client = RailScoreClient(api_key=RAIL_API_KEY)

## 1. GDPR Compliance

In [ ]:
gdpr_result = client.compliance_check(
    content=(
        "Our AI model processes user browsing history and purchase patterns "
        "to generate personalized product recommendations. We collect IP "
        "addresses, device fingerprints, and cross-site tracking data "
        "without explicit consent."
    ),
    framework="gdpr",
    context={
        "domain": "e-commerce",
        "data_types": ["browsing_history", "purchase_data", "ip_address"],
        "processing_purpose": "personalized_recommendations",
    },
)

print(f"Score: {gdpr_result.compliance_score.score}/10 ({gdpr_result.compliance_score.label})")
print(f"Summary: {gdpr_result.compliance_score.summary}")
print(f"Requirements: {gdpr_result.requirements_passed}/{gdpr_result.requirements_checked} passed")

In [ ]:
if gdpr_result.issues:
    print("Issues found:")
    for issue in gdpr_result.issues[:5]:
        print(f"  [{issue.severity.upper()}] {issue.description}")
        print(f"    Article: {issue.article} | Effort: {issue.remediation_effort}")

## 2. Multi-Framework Check (GDPR + CCPA)

In [ ]:
multi_result = client.compliance_check(
    content=(
        "We use cookies to track user behavior across websites and sell "
        "aggregated profiles to third-party advertisers. Users can opt out "
        "via a link buried in our privacy policy footer."
    ),
    frameworks=["gdpr", "ccpa"],
    context={
        "domain": "advertising",
        "data_types": ["cookies", "behavioral_data", "user_profiles"],
    },
)

summary = multi_result.cross_framework_summary
print(f"Frameworks evaluated: {summary.frameworks_evaluated}")
print(f"Average score: {summary.average_score}/10")
print(f"Weakest: {summary.weakest_framework} ({summary.weakest_score}/10)")

for fw_name, fw_result in multi_result.results.items():
    cs = fw_result.compliance_score
    print(f"\n  {fw_name.upper()}: {cs.score}/10 ({cs.label})")
    print(f"    {fw_result.requirements_passed}/{fw_result.requirements_checked} passed")

## 3. HIPAA (Healthcare)

In [ ]:
hipaa_result = client.compliance_check(
    content=(
        "Patient records are stored in encrypted databases with access "
        "controls. All staff undergo annual privacy training. Patient "
        "information is only shared with authorized healthcare providers "
        "for treatment purposes."
    ),
    framework="hipaa",
    context={
        "domain": "healthcare",
        "data_types": ["phi"],
        "processing_purpose": "treatment",
    },
)

print(f"Score: {hipaa_result.compliance_score.score}/10 ({hipaa_result.compliance_score.label})")
print(f"Requirements: {hipaa_result.requirements_passed}/{hipaa_result.requirements_checked} passed")

if hipaa_result.improvement_suggestions:
    print("\nSuggestions:")
    for s in hipaa_result.improvement_suggestions[:3]:
        print(f"  - {s}")

## 4. EU AI Act — Risk Classification

In [ ]:
ai_act_result = client.compliance_check(
    content=(
        "Our facial recognition system is deployed in public spaces for "
        "real-time surveillance. It identifies individuals without their "
        "knowledge. No risk assessment was conducted before deployment."
    ),
    framework="eu_ai_act",
    context={
        "domain": "law_enforcement",
        "system_type": "biometric_identification",
        "data_types": ["biometric_data", "facial_images"],
        "risk_indicators": [
            "real_time_surveillance",
            "biometric_identification",
            "law_enforcement",
        ],
    },
)

print(f"Score: {ai_act_result.compliance_score.score}/10 ({ai_act_result.compliance_score.label})")

if ai_act_result.risk_classification_detail:
    rd = ai_act_result.risk_classification_detail
    print(f"Risk Tier: {rd.tier}")
    print(f"Basis: {rd.basis}")

## 5. Strict Mode

Strict mode raises the compliance threshold from 7.0 to 8.5.

In [ ]:
strict_result = client.compliance_check(
    content=(
        "Our AI chatbot provides general customer service. It uses "
        "anonymized data and identifies itself as AI. Users can request "
        "human support. Conversations are encrypted and deleted after 30 days."
    ),
    framework="ccpa",
    strict_mode=True,
    context={"domain": "customer_service"},
)

print(f"Score: {strict_result.compliance_score.score}/10 ({strict_result.compliance_score.label})")
print(f"Strict mode threshold: 8.5")
print(f"Requirements: {strict_result.requirements_passed}/{strict_result.requirements_checked} passed")
print(f"Warnings: {strict_result.requirements_warned}")